# 일원 분산 분석

주어진 데이터는 4가지 다른 교육 방법을 적용한 대학생들의 학점 결과이다. 이 실험에서는 비슷한 실력을 가진 학생 40명을 무작위로 4개(A, B, C, D)그룹으로 나누었고, 각 그룹은 다른 교육 방법을 적용했다. 학생들의 학점 결과에는 교육 방법에 따른 차이가 있는지 유의수준 0.05하에서 검정하시오.
- 귀무가설(H0): 네 가지 교육 방법에 의한 학생들의 학점 평균은 동일하다.
- 대립가설(H1): 적어도 두 그룹의 학점 평균은 다르다.

## 1. 기초

In [1]:
import pandas as pd
df = pd.DataFrame({
    'A': [3.5, 4.3, 3.8, 3.6, 4.1, 3.2, 3.9, 4.4, 3.5, 3.3],
    'B': [3.9, 4.4, 4.1, 4.2, 4.5, 3.8, 4.2, 3.9, 4.4, 4.3],
    'C': [3.2, 3.7, 3.6, 3.9, 4.3, 4.1, 3.8, 3.5, 4.4, 4.0],
    'D': [3.8, 3.4, 3.1, 3.5, 3.6, 3.9, 3.2, 3.7, 3.3, 3.4]
})
print(df.head(2))

     A    B    C    D
0  3.5  3.9  3.2  3.8
1  4.3  4.4  3.7  3.4


### 일원 분산 분석

In [2]:
# 일원 분산 분석
from scipy import stats

stats.f_oneway(df['A'], df['B'], df['C'], df['D'])

F_onewayResult(statistic=np.float64(7.296983758700697), pvalue=np.float64(0.0006053225519892243))

In [4]:
# 정규성, 등분산, 일원 분산 분석

# Shapiro-Wilk(샤피로-윌크) 정규성 검정
print(stats.shapiro(df['A']))
print(stats.shapiro(df['B']))
print(stats.shapiro(df['C']))
print(stats.shapiro(df['D']))

# Levene(레빈) 등분산 검정
print(stats.levene(df['A'], df['B'], df['C'], df['D']))

# 일원 분산 분석
print(stats.f_oneway(df['A'], df['B'], df['C'], df['D']))

ShapiroResult(statistic=np.float64(0.9498823422157768), pvalue=np.float64(0.667108682193412))
ShapiroResult(statistic=np.float64(0.9346447100990081), pvalue=np.float64(0.495096931740429))
ShapiroResult(statistic=np.float64(0.9871342658078467), pvalue=np.float64(0.9919546177620264))
ShapiroResult(statistic=np.float64(0.9752339025839639), pvalue=np.float64(0.9346854448707619))
LeveneResult(statistic=np.float64(1.5433829973707245), pvalue=np.float64(0.22000894224209636))
F_onewayResult(statistic=np.float64(7.296983758700697), pvalue=np.float64(0.0006053225519892243))


## 2. 심화

In [8]:
# 데이터 재구조화 (긴 형태)
df_melt = df.melt()
df_melt.head()

,variable,value
0,A,3.5
1,A,4.3
2,A,3.8
3,A,3.6
4,A,4.1


### 분산분석 테이블

In [9]:
# ANOVA 테이블
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('value ~ variable', data=df_melt).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
variable,3.0,2.35875,0.78625,7.296984,0.000605
Residual,36.0,3.87900,0.10775,NaN,NaN


### 사후검정
- 목적: 어떤 그룹들 간에 통계적으로 유의미한 차이가 있는지 구체적으로 파악하는 것

In [10]:
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison

# Tukey HSD (투키)
tukey_result = pairwise_tukeyhsd(df_melt['value'], df_melt['variable'], alpha=0.05)
print(tukey_result.summary())

# Bonferroni(본페로니)
mc = MultiComparison(df_melt['value'], df_melt['variable'])
bon_result = mc.allpairtest(stats.ttest_ind, method='bonf')
print(bon_result[0])

Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     A      B     0.41 0.0397  0.0146  0.8054   True
     A      C     0.09 0.9273 -0.3054  0.4854  False
     A      D    -0.27 0.2722 -0.6654  0.1254  False
     B      C    -0.32 0.1483 -0.7154  0.0754  False
     B      D    -0.68 0.0003 -1.0754 -0.2846   True
     C      D    -0.36 0.0852 -0.7554  0.0354  False
----------------------------------------------------
Test Multiple Comparison ttest_ind 
FWER=0.05 method=bonf
alphacSidak=0.01, alphacBonf=0.008
group1 group2   stat   pval  pval_corr reject
---------------------------------------------
     A      B -2.7199  0.014    0.0843  False
     A      C  -0.515 0.6128       1.0  False
     A      D  1.7538 0.0965    0.5788  False
     B      C  2.2975 0.0338    0.2028  False
     B      D  6.0686    0.0    0.0001   True
     C      D  2.5219 0.0213    0.1279  False
---------

### 크루스칼-왈리스 검정 (비모수 검정)

In [12]:
import pandas as pd
from scipy import stats

# 데이터
df = pd.DataFrame({
    'A': [10.5, 11.3, 10.8, 10.6, 11.1, 10.2, 10.9, 11.4, 10.5, 10.3],
    'B': [10.9, 11.4, 11.1, 11.2, 11.5, 10.8, 11.2, 10.9, 11.4, 11.3],
    'C': [10.2, 10.7, 10.6, 10.9, 11.3, 11.1, 10.8, 10.5, 11.4, 11.0],
    'D': [13.8, 10.4, 10.1, 10.5, 10.6, 10.9, 10.2, 10.7, 10.3, 10.4]
})

# 정규성 검정
print(stats.shapiro(df['A']))
print(stats.shapiro(df['B']))
print(stats.shapiro(df['C']))
print(stats.shapiro(df['D']))

# Kruskal-Wallis 검정
stats.kruskal(df['A'], df['B'], df['C'], df['D'])

ShapiroResult(statistic=np.float64(0.9498823422157767), pvalue=np.float64(0.6671086821934112))
ShapiroResult(statistic=np.float64(0.9346447100990083), pvalue=np.float64(0.49509693174043135))
ShapiroResult(statistic=np.float64(0.9871342658078467), pvalue=np.float64(0.9919546177620264))
ShapiroResult(statistic=np.float64(0.5759975003447693), pvalue=np.float64(2.8656624682936833e-05))


KruskalResult(statistic=np.float64(11.183607021517561), pvalue=np.float64(0.010773365310213669))